# Week 10 — Day 1 : Model Context Protocol (MCP)

**Bootcamp GenAI & Machine Learning 2026 — Developers Institute**

## Goal

Build a tiny MCP server (one tool + one resource template) and a Python client that connects over **STDIO**, discovers features, and invokes them.

## Architecture

```
┌─────────────────────────────────────────────────┐
│                  MCP Host                        │
│  ┌─────────────┐      ┌──────────────────────┐  │
│  │   client.py  │◄────►│     server.py        │  │
│  │ ClientSession│ STDIO│  FastMCP("Demo")     │  │
│  │              │      │  ├── tool: add()     │  │
│  │              │      │  └── resource:       │  │
│  │              │      │      greeting://{name}│  │
│  └─────────────┘      └──────────────────────┘  │
└─────────────────────────────────────────────────┘
```

| Component | Role |
|-----------|------|
| **MCP Server** (`server.py`) | Exposes tools and resources over STDIO |
| **MCP Client** (`client.py`) | Discovers and invokes server features |
| **STDIO transport** | Spawns the server as a subprocess — zero network config |

---
## Setup

In [ ]:
# Install the MCP package (includes CLI)
%pip install -q "mcp[cli]"

---
## A. Server — `server.py`

The server exposes:
- **Tool** `add(a, b)` — returns the sum of two integers
- **Resource template** `greeting://{name}` — returns a personalized greeting

In [ ]:
# server.py — display the file content
with open("server.py") as f:
    print(f.read())

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Returns the sum of two integers."""
    return a + b


@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Returns a personalized greeting for the given name."""
    return f"Hello, {name}!"


if __name__ == "__main__":
    mcp.run()

### Key concepts — Server side

| Decorator | What it does |
|-----------|-------------|
| `@mcp.tool()` | Registers a callable action — clients can invoke it with arguments |
| `@mcp.resource("greeting://{name}")` | Registers a **URI template** — `{name}` is substituted when the client reads a concrete URI |
| `mcp.run()` | Starts the STDIO event loop — reads JSON-RPC from stdin, writes to stdout |

---
## B. Client — `client.py`

The client:
1. Spawns `server.py` as a subprocess via the `mcp` CLI
2. Opens a `ClientSession` over STDIO
3. Lists resource templates and tools
4. Reads `greeting://hello`
5. Calls `add(a=1, b=7)`

In [ ]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Spawn server.py via the mcp CLI over STDIO
server_params = StdioServerParameters(
    command="mcp", args=["run", "server.py"], env=None
)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:

            # 1. Initialise the session
            await session.initialize()
            print("Session initialized ✅\n")

            # 2. List resource templates (greeting://{name} is a template)
            templates = await session.list_resource_templates()
            print("=== Resource Templates ===")
            if templates.resourceTemplates:
                for t in templates.resourceTemplates:
                    print(f"  - {t.name}: {t.uriTemplate}")
            else:
                print("  (none)")

            # Also list any concrete resources
            resources = await session.list_resources()
            print("\n=== Concrete Resources ===")
            if resources.resources:
                for r in resources.resources:
                    print(f"  - {r.name}: {r.uri}")
            else:
                print("  (none — greeting is a template)")

            # 3. List tools
            tools = await session.list_tools()
            print("\n=== Tools ===")
            for t in tools.tools:
                print(f"  - {t.name}: {t.description}")

            # 4. Read greeting://hello (substitutes {name} = "hello")
            content = await session.read_resource("greeting://hello")
            print("\n=== Read greeting://hello ===")
            print(f"  {content.contents[0].text}")

            # 5. Call the add tool with a=1, b=7
            result = await session.call_tool("add", {"a": 1, "b": 7})
            print("\n=== Call add(a=1, b=7) ===")
            print(f"  Result: {result.content[0].text}")


if __name__ == "__main__":
    asyncio.run(run())

### Key concepts — Client side

| Concept | Description |
|---------|------------|
| `StdioServerParameters` | Tells the client how to spawn the server subprocess |
| `stdio_client(server_params)` | Opens STDIO pipes to the server process |
| `ClientSession` | JSON-RPC session that handles the MCP protocol handshake |
| `session.initialize()` | Performs the MCP capability negotiation |
| `list_resource_templates()` | Returns URI templates registered on the server |
| `list_tools()` | Returns callable tools registered on the server |
| `read_resource(uri)` | Resolves a URI template with concrete values and returns content |
| `call_tool(name, args)` | Invokes a server tool with the given arguments |

---
## C. Run

The client spawns the server automatically over STDIO:

```bash
python client.py
```

Or in two separate terminals:
```bash
# Terminal 1
mcp run server.py

# Terminal 2
python client.py
```

In [ ]:
# Run the client (spawns server automatically)
import subprocess, sys
result = subprocess.run(
    [sys.executable, "client.py"],
    capture_output=True, text=True, timeout=30
)
print(result.stdout)
if result.stderr:
    # Filter out INFO log lines from the server (expected)
    errors = [l for l in result.stderr.splitlines() if "ERROR" in l or "Exception" in l]
    if errors:
        print("STDERR:", "\n".join(errors))

---
## Terminal Output Capture

```
Session initialized ✅

=== Resource Templates ===
  - greet: greeting://{name}

=== Concrete Resources ===
  (none — greeting is a template)

=== Tools ===
  - add: Returns the sum of two integers.

=== Read greeting://hello ===
  Hello, hello!

=== Call add(a=1, b=7) ===
  Result: 8
```

**Observations:**
- `greeting://{name}` appears as a **Resource Template** (not a concrete resource) — the URI template is resolved at read time
- Reading `greeting://hello` substitutes `name = "hello"` → returns `"Hello, hello!"`
- `add(1, 7)` returns `8` ✅
- STDIO transport is zero-config for local development — no ports, no auth, no network

---
## MCP Concepts Summary

| Concept | Description |
|---------|------------|
| **Host** | The application that manages MCP clients (e.g. Claude Desktop, your script) |
| **Client** | One connection to one server; handles capability negotiation |
| **Server** | Exposes tools and resources; runs as a subprocess or network service |
| **Tool** | A callable action with typed arguments → LLMs can invoke these |
| **Resource** | Read-only context (files, DB rows, API results) identified by a URI |
| **Resource Template** | A URI pattern with `{variable}` placeholders — resolved at read time |
| **STDIO transport** | Server is a subprocess; client communicates via stdin/stdout — ideal for local dev |
| **JSON-RPC 2.0** | The wire protocol used by MCP for all messages |

### Why STDIO for local dev?

- **No port conflicts** — server process is isolated per client connection
- **No auth setup** — subprocess inherits environment/permissions from parent
- **Easy debugging** — server logs go to stderr, visible in your terminal
- **Simple lifecycle** — server starts when client connects, stops when client disconnects